In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-5 ; m=1; l0=1 ; k=2 ; g=9.8 ;
tf=15; N=int(tf/dt);

t=np.linspace(0,tf,N)


r_0= 2             #condiciones iniciales
r_dot_0 = 0
theta_0= np.pi/3
w_0= 0

r0 = l0 + m*g/k              # longitud de equilibrio estático
wp = np.sqrt(g/r0)           # frecuencia angular del péndulo
ws= np.sqrt(k/m)              # frecuencia del resorte

## FUNCION PARA EL MOVIMIENTO


In [ ]:
def Motion(r0,r_0, r_dot0, theta0, w0, ws, wp, t):
    N = len(t)
    r = np.zeros(N)
    r_dot = np.zeros(N)
    r_dotdot = np.zeros(N)
    
    theta = np.zeros(N)
    w = np.zeros(N)
    w_dot = np.zeros(N)

    # Condiciones iniciales
    r[0] = r_0
    r_dot[0] = r_dot_0
    theta[0] = theta_0
    w[0] = w_0

    
    la= (r/r0) - 1
    la_dot= r_dot/r0

    for i in range(N - 1):
        dt = t[i+1] - t[i]
        # --- PREDICTOR (Euler) ---
        ala_pred = ala_num(la[i], la_dot[i], theta[i], w[i], ws, wp)
        ath_pred = ath_num(la[i], la_dot[i], theta[i], w[i], ws, wp)

        la_dot_pred = la_dot[i] + ala_pred * dt
        la_pred     = la[i]     + la_dot[i] * dt
        w_pred      = w[i]      + ath_pred * dt
        theta_pred  = theta[i]  + w[i] * dt

        # --- Aceleraciones en el punto predicho ---
        ala_next = ala_num(la_pred, la_dot_pred, theta_pred, w_pred, ws, wp)
        ath_next = ath_num(la_pred, la_dot_pred, theta_pred, w_pred, ws, wp)

        # --- CORRECTOR (Heun) ---
        la[i+1]     = la[i]     + 0.5 * (la_dot[i] + la_dot_pred) * dt
        theta[i+1]  = theta[i]  + 0.5 * (w[i] + w_pred) * dt
        la_dot[i+1] = la_dot[i] + 0.5 * (ala_pred + ala_next) * dt
        w[i+1]      = w[i]      + 0.5 * (ath_pred + ath_next) * dt
        
    return la, theta, la_dot, w

In [ ]:
la , la_dot, theta, w = sp.symbols('la la_dot theta w' , real=True)
r, r_dot = sp.symbols('r r_dot', real=True)
ws_ , wp_ = sp.symbols('ws_ wp_', positive=True)

ala_sym = (la+1)*w**2 - la*ws_**2 - (1-sp.cos(theta))*wp_**2
ath_sym = -(1 / (la + 1 )) * (sp.sin(theta)*wp_**2 + 2*la_dot*w )

ala_num =sp.lambdify((la,la_dot,theta,w,ws_,wp_), ala_sym, 'numpy')
ath_num =sp.lambdify((la,la_dot,theta,w,ws_,wp_), ath_sym, 'numpy')

In [ ]:
#Simulación
la_sol, theta_sol, la_dot_sol, w_sol = Motion(r0,r_0, r_dot_0, theta_0, w_0, ws, wp, t)

## Lagrangiano, Hamiltoniano y Energía

In [ ]:
#Lagrangiano, Hamiltoniano y Energía


r_s, rd_s, th_s, thd_s = sp.symbols('r rdot theta thetadot', real=True)
m_, k_, l0_, g_ = sp.symbols('m k l0 g', positive=True)

T_sym = sp.Rational(1,2)*m_*(rd_s**2 + r_s**2*thd_s**2)
U_sym = sp.Rational(1,2)*k_*(r_s - l0_)**2 - m_*g_*r_s*sp.cos(th_s)
L_sym = T_sym - U_sym
E_sym = T_sym + U_sym

pr_sym  = sp.diff(L_sym, rd_s)                   # p_r  = ∂L/∂ṙ   (derivada parcial)
pth_sym = sp.diff(L_sym, thd_s)                  # p_θ  = ∂L/∂θ̇
H_sym   = sp.simplify(pr_sym*rd_s + pth_sym*thd_s - L_sym)

args = (r_s, rd_s, th_s, thd_s, m_, k_, l0_, g_)
L_num, E_num, H_num, pr_num, pth_num = (sp.lambdify(args, f, 'numpy')
                                        for f in (L_sym, E_sym, H_sym, pr_sym, pth_sym))

# Variables físicas
r_sol  = (la_sol + 1)*r0
rd_sol = la_dot_sol*r0

pars = (m, k, l0, g)
L  = L_num(r_sol, rd_sol, theta_sol, w_sol, *pars)
E  = E_num(r_sol, rd_sol, theta_sol, w_sol, *pars)
H  = H_num(r_sol, rd_sol, theta_sol, w_sol, *pars)
pr = pr_num(r_sol, rd_sol, theta_sol, w_sol, *pars)
pth = pth_num(r_sol, rd_sol, theta_sol, w_sol, *pars)

print('H simbólico:', H_sym)
print('max |E - H|         =', np.abs(E - H).max())
print('max |E - E0| / |E0| =', np.abs(E - E[0]).max()/abs(E[0]))

In [ ]:
#Equilibrio: ∂U/∂r = 0 y ∂U/∂θ = 0  ->  θ = 0, r = l0 + m g / k
r_eq, th_eq = r0, 0.0
print(f'r_eq = {r_eq:.4f} m,  theta_eq = {th_eq}')

# Comprobación: las aceleraciones en el equilibrio deben ser 0
print('ala(eq) =', ala_num(0.0, 0.0, 0.0, 0.0, ws, wp), ' ath(eq) =', ath_num(0.0, 0.0, 0.0, 0.0, ws, wp))

# Pequeñas oscilaciones: modo radial (resorte) y modo angular (péndulo)
Om_r, Om_th = ws, wp
print(f'Modo radial : Omega = {Om_r:.4f} rad/s,  T = {2*np.pi/Om_r:.4f} s')
print(f'Modo angular: Omega = {Om_th:.4f} rad/s,  T = {2*np.pi/Om_th:.4f} s')

In [ ]:
#Plot Lagrangiano, Hamiltoniano y Energía
def plot_LHE(t, L, H, E, s=None):
    s = s or max(1, len(t)//20000)                 # submuestreo
    fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

    ax[0].plot(t[::s], L[::s], color='tab:purple')
    ax[0].set(ylabel='L (J)', title='Lagrangiano')

    ax[1].plot(t[::s], E[::s], color='tab:blue', lw=3, label='E = T + U')
    ax[1].plot(t[::s], H[::s], 'r--', lw=1.5, label='H = p_r ṙ + p_θ θ̇ − L')
    ax[1].ticklabel_format(useOffset=False)
    ax[1].set_ylim(E[0] - 0.5*abs(E[0]), E[0] + 0.5*abs(E[0]))
    ax[1].set(ylabel='J', title='E = H ?'); ax[1].legend()

    eps = 1e-18                                    # evita log(0)
    ax[2].semilogy(t[::s], np.abs(E[::s] - E[0])/abs(E[0]) + eps, color='tab:blue',  lw=1, label='|E − E₀| / |E₀|  (integrador)')
    ax[2].semilogy(t[::s], np.abs(H[::s] - E[::s])/abs(E[0]) + eps, color='tab:green', lw=1, label='|H − E| / |E₀|  (redondeo)')
    ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8)

    for A in ax: A.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_LHE(t, L, H, E)

In [ ]:
#Animación
plt.rcParams['animation.embed_limit'] = 60         # MB; el límite por defecto (20) se queda corto

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 30       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 3.0      # duración de la estela de la masa (s); 0 = sin estela
n_coils = 12       # vueltas del muelle dibujado

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

rs, ths = r_sol[idx], theta_sol[idx]
xm, ym  = rs*np.sin(ths), -rs*np.cos(ths)          # masa (pivote en el origen, y hacia arriba)

def spring_xy(x1, y1, n=n_coils, amp=0.05*rs.max()):
    """Zigzag que une el pivote (0,0) con la masa (x1,y1)."""
    d = np.hypot(x1, y1)
    u = np.array([x1, y1])/d                       # vector unitario a lo largo del muelle
    nrm = np.array([-u[1], u[0]])                  # normal
    sgrid = np.linspace(0, 1, 2*n + 1)
    off = amp*np.where(np.arange(2*n + 1) % 2 == 1, 1.0, -1.0)
    off[0] = off[-1] = 0                           # extremos rectos
    pts = np.outer(sgrid*d, u) + np.outer(off, nrm)
    return pts[:, 0], pts[:, 1]

# Figura
fig, ax = plt.subplots(figsize=(6, 6))
Lm = rs.max()*1.1
ax.set(xlim=(-Lm, Lm), ylim=(min(ym.min(), -0.2*Lm) - 0.1*Lm, max(ym.max(), 0) + 0.2*Lm), aspect='equal',
       xlabel='x (m)', ylabel='y (m)', title='Masa en un muelle en un plano vertical')
ax.grid(alpha=0.3)
ax.plot(0, 0, 'ks', ms=8)                          # punto de suspensión

spring, = ax.plot([], [], 'k-', lw=1.2)
bob,    = ax.plot([], [], 'o', color='tab:red', ms=10)
trail,  = ax.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)
time_txt = ax.text(0.03, 0.95, '', transform=ax.transAxes)

def update(k):
    spring.set_data(*spring_xy(xm[k], ym[k]))
    bob.set_data([xm[k]], [ym[k]])
    if trail_n > 0:
        j = max(0, k - trail_n)
        trail.set_data(xm[j:k+1], ym[j:k+1])
    time_txt.set_text(f't = {t[idx[k]]:.2f} s')
    return spring, bob, trail, time_txt

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
#Comparación con la solución teórica de pequeñas oscilaciones
#   λ(t) = A cos(ωs t),   θ(t) = B sin(ωp t)      (λ = r/r0 - 1)
A, B = 0.01, 0.01                 # amplitudes pequeñas: λ adimensional, θ en rad
tf_s, dt_s = 30, 5e-4             # simulación corta (basta con unos periodos)
t_s = np.arange(0, tf_s, dt_s)

# Condiciones iniciales del enunciado: θ0 = 0, λ0 = A, θ̇0 = ωp·B, λ̇0 = 0
# (Motion lee r_0, r_dot_0, theta_0, w_0 como globales: las fijamos y luego las restauramos)
_bak = (r_0, r_dot_0, theta_0, w_0)
r_0, r_dot_0, theta_0, w_0 = r0*(1 + A), 0.0, 0.0, wp*B
la_n, th_n, lad_n, w_n = Motion(r0, r_0, r_dot_0, theta_0, w_0, ws, wp, t_s)
r_0, r_dot_0, theta_0, w_0 = _bak

# Solución teórica (orden más bajo)
la_t = A*np.cos(ws*t_s)
th_t = B*np.sin(wp*t_s)

# Gráficas
fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharex=True)

ax[0,0].plot(t_s, la_n, lw=2, label='numérica')
ax[0,0].plot(t_s, la_t, 'r--', lw=1.2, label='teórica  A cos(ωs t)')
ax[0,0].set(ylabel='λ', title='Modo radial'); ax[0,0].legend()

ax[0,1].plot(t_s, th_n, lw=2, label='numérica')
ax[0,1].plot(t_s, th_t, 'r--', lw=1.2, label='teórica  B sin(ωp t)')
ax[0,1].set(ylabel='θ (rad)', title='Modo angular'); ax[0,1].legend()

ax[1,0].plot(t_s, la_n - la_t, color='tab:green')
ax[1,0].set(xlabel='t (s)', ylabel='λ_num − λ_teo')

ax[1,1].plot(t_s, th_n - th_t, color='tab:green')
ax[1,1].set(xlabel='t (s)', ylabel='θ_num − θ_teo')

for a in ax.ravel(): a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'max |λ_num − λ_teo| / A = {np.abs(la_n - la_t).max()/A:.2e}')
print(f'max |θ_num − θ_teo| / B = {np.abs(th_n - th_t).max()/B:.2e}')

In [ ]:
#Comprobación: el error de la aproximación lineal debe crecer al aumentar la amplitud
print(f'{"A=B":>8} {"err λ / A":>12} {"err θ / B":>12}')
for amp in (0.005, 0.01, 0.02, 0.04, 0.08):
    _bak = (r_0, r_dot_0, theta_0, w_0)
    r_0, r_dot_0, theta_0, w_0 = r0*(1 + amp), 0.0, 0.0, wp*amp
    la_n, th_n, _, _ = Motion(r0, r_0, r_dot_0, theta_0, w_0, ws, wp, t_s)
    r_0, r_dot_0, theta_0, w_0 = _bak
    e_la = np.abs(la_n - amp*np.cos(ws*t_s)).max()/amp
    e_th = np.abs(th_n - amp*np.sin(wp*t_s)).max()/amp
    print(f'{amp:8.3f} {e_la:12.2e} {e_th:12.2e}')